# Agri-Advisor Module 4: Leaf Disease Detection (Colab / Kaggle GPU)

**Before running:** Runtime -> Change runtime type -> **GPU** (T4 is fine).

Steps: (1) get the project code, (2) download PlantVillage, (3) train, (4) optional real-field check with PlantDoc, (5) inspect results + Grad-CAM, (6) download the trained model.

Expect roughly 20-40 min on a T4 for EfficientNet-B0. Times vary; check the per-epoch log.

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 1. Get the project code
Upload `agri-advisor.zip` (the file from this conversation) when prompted.

In [ ]:
from google.colab import files
up = files.upload()            # choose agri-advisor.zip
!unzip -q -o agri-advisor.zip
%cd agri-advisor
!pip -q install kagglehub scikit-learn matplotlib pillow

## 2. Download PlantVillage
Uses the public Kaggle dataset `abdallahalidev/plantvillage-dataset`. If `kagglehub` asks for credentials, create a Kaggle API token (kaggle.com -> Settings -> API) or upload the dataset manually. **Check the printed folder layout**: we want the `color` folder (one sub-folder per class).

In [ ]:
import kagglehub, glob, os
path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
print('downloaded to', path)
cands = [p for p in glob.glob(path + '/**/color', recursive=True) if os.path.isdir(p)]
assert cands, "No 'color' folder found - inspect the layout above and set DATA_DIR by hand"
DATA_DIR = cands[0]
classes = sorted(os.listdir(DATA_DIR))
print(len(classes), 'classes, e.g.', classes[:3])
print('images:', sum(len(os.listdir(os.path.join(DATA_DIR, c))) for c in classes))

## 3. (Recommended) Get PlantDoc for a real-field check

PlantVillage test accuracy is inflated (near-duplicate lab photos on plain backgrounds). PlantDoc has field photos, so scoring on it shows the real lab-to-field gap. Folder names differ from PlantVillage, so only classes matchable by name are scored, and the script prints what it skipped. To include more, pass `--field_map map.json` with `{"<PlantDoc folder>": "<PlantVillage class>"}`.

**Verify the printed layout**: we want the folder containing one sub-folder per class (usually `PlantDoc-Dataset/test`). Set `FIELD_ARG = ""` to skip.

In [ ]:
!git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset
!ls PlantDoc-Dataset && ls PlantDoc-Dataset/test | head -5
FIELD_ARG = "--field_dir PlantDoc-Dataset/test"

## 4. Train (once)
Two phases: head-only (3 epochs), then full fine-tune (up to 12, early stopping). Use `--arch mobilenet_v3_large` for a smaller model that suits phones.

In [ ]:
!python src/disease_train.py --data_dir "$DATA_DIR" $FIELD_ARG --arch efficientnet_b0 \
    --epochs_head 3 --epochs_finetune 12 --batch_size 64 --num_workers 2 --out_dir models/disease

## 5. Inspect results

In [ ]:
import json
m = json.load(open('models/disease/metrics.json'))
for k in ['test_acc','test_top3','test_macro_f1','temperature','ece_uncalibrated','ece_calibrated','field_eval']:
    print(f'{k:18s}', m[k])
print('\nMost common confusions:')
for c in m['top_confusions'][:8]: print(' ', c)
from IPython.display import Image, display
display(Image('models/disease/confusion_matrix.png'))

In [ ]:
# Grad-CAM on any leaf photo (upload one, or point at a dataset image)
import random
img = random.choice(glob.glob(DATA_DIR + '/*/*.JPG') + glob.glob(DATA_DIR + '/*/*.jpg'))
print(img)
!python src/disease_predict.py "$img" --ckpt models/disease/disease_model.pt --gradcam gradcam.png
display(Image('gradcam.png'))

## 6. Download the trained model
Put `disease_model.pt` into `models/disease/` in your local project; the API then serves `POST /predict/disease`.

In [ ]:
!zip -q -r disease_model_bundle.zip models/disease
files.download('disease_model_bundle.zip')